In [76]:
using Gridap
using Gridap.FESpaces
using Gridap.ReferenceFEs
using Gridap.Arrays
using Gridap.Geometry
using Gridap.Fields
using Gridap.CellData
using FillArrays
using Test
using InteractiveUtils
using Gridap
using Gridap.Fields
using Gridap.Polynomials
using Gridap.ReferenceFEs
using LinearAlgebra
using FillArrays
using StaticArrays

In [ ]:
# -----------------------------
# Material parameters
# -----------------------------
E = 119e3
ν = 0.3
C = (E / (1 - ν^2)) * [
    1.0  ν   0.0;
    ν    1.0 0.0;
    0.0  0.0 (1-ν)/2
]

# -----------------------------
# Mesh
# -----------------------------
L, W = 60.0, 20.0
partition = (Int(L*3), Int(W*3))
model = CartesianDiscreteModel((0.0,L,0.0,W), partition)

labels = get_face_labeling(model)
add_tag_from_tags!(labels, "left", [1,3,7])
add_tag_from_tags!(labels, "right", [2,4,8])


tria = Triangulation(model)
ncells = num_cells(tria)
cell_coords = get_cell_coordinates(tria)

# -----------------------------
# Shape functions (bilinear)
# -----------------------------
ref_nodes = Point{2,Float64}[(0,0),(1,0),(0,1),(1,1)]
filter(e,p) = true
m = MonomialBasis{2}(Float64,1,filter)
l = LagrangianDofBasis(Float64,ref_nodes)
change = inv(evaluate(l,m))
s = linear_combination(change,m)

# Quadrature (1-point)
q = [Point{2,Float64}((0.5,0.5))]
w = [1.0]

cell_s = Fill(s, ncells)
cell_q = Fill(q, ncells)
cell_w = Fill(w, ncells)

cell_φ = lazy_map(linear_combination, cell_coords, cell_s)
cell_Jt = lazy_map(∇, cell_φ)
cell_detJ = lazy_map(Broadcasting(det), cell_Jt)
cell_invJt = lazy_map(Operation(inv), cell_Jt)
cell_∇ref_s = lazy_map(Broadcasting(∇), cell_s)
cell_∇s = lazy_map(Broadcasting(Operation(⋅)), cell_invJt, cell_∇ref_s)
# Cell-wise stifness matrix
cell_∇st = lazy_map(transpose,cell_∇s)
cell_∇s∇st = lazy_map( Broadcasting(Operation(⋅)),cell_∇s,cell_∇st)
# -----------------------------
# B-matrix
# -----------------------------
function B_matrix(∇s_q)
    nshape = length(∇s_q)
    B = zeros(3,2*nshape)
    for i in 1:nshape
        dNdx = ∇s_q[i]
        B[1,2i-1] = dNdx[1]
        B[2,2i]   = dNdx[2]
        B[3,2i-1] = dNdx[2]
        B[3,2i]   = dNdx[1]
    end
    return B
end
# # -----------------------------
function Ke_func(∇s_cell, detJ_cell, wq)
    ∇s_q = [evaluate(∇s_cell[i], q[1]) for i in 1:length(∇s_cell)]
    B = B_matrix(∇s_q)
    return B' * C * B * abs(detJ_cell(q[1])) * wq[1]
end
Ke_collection = lazy_map(Ke_func, cell_∇s, cell_detJ, cell_w)
# -----------------------------
n_nodes = num_nodes(model)
ndofs = 2*n_nodes
K_global = zeros(ndofs, ndofs)
F_global = zeros(ndofs)
# -----------------------------
# Assemble global stiffness
# -----------------------------
conn = get_cell_node_ids(tria)
for (icell, Ke) in enumerate(Ke_collection)
    nodes = conn[icell]
    for i in 1:4, j in 1:4
        K_global[2*nodes[i]-1, 2*nodes[j]-1] += Ke[2*i-1, 2*j-1]
        K_global[2*nodes[i]-1, 2*nodes[j]]   += Ke[2*i-1, 2*j]
        K_global[2*nodes[i],   2*nodes[j]-1] += Ke[2*i,   2*j-1]
        K_global[2*nodes[i],   2*nodes[j]]   += Ke[2*i,   2*j]
    end
end


In [78]:
# -----------------------------
# Load vector (right edge)
# -----------------------------
Γ = BoundaryTriangulation(model, tags="right")
face_node_ids = get_cell_node_ids(Γ)
face_coords   = get_cell_coordinates(Γ)

# Linear 1D shape
ref_nodes_line = Point{1,Float64}[Point(0.0), Point(1.0)]
m_line = MonomialBasis{1}(Float64,1)
l_line = LagrangianDofBasis(Float64, ref_nodes_line)
change_line = inv(evaluate(l_line,m_line))
s_line = linear_combination(change_line,m_line)
q_line = [Point{1,Float64}((0.5,))]
w_line = [1.0]
t = SVector(0.0, -1.0)

for iface in 1:length(face_node_ids)
    nodes = face_node_ids[iface]
    X_face = face_coords[iface]
    N_vals = [evaluate(s_line[i], q_line[1]) for i in 1:2]
    J = norm(X_face[2] - X_face[1])
    Fe = zeros(4)
    for i in 1:2
        Fe[2i-1] = t[1] * N_vals[i] * J * w_line[1]
        Fe[2i]   = t[2] * N_vals[i] * J * w_line[1]
    end
    for i in 1:2
        dof_x = 2*nodes[i]-1
        dof_y = 2*nodes[i]
        F_global[dof_x] += Fe[2i-1]
        F_global[dof_y] += Fe[2i]
    end
end

# -----------------------------
# Apply Dirichlet BC (left edge fixed)
# -----------------------------
Γ_left = BoundaryTriangulation(model, tags="left")
left_nodes = unique(vcat(get_cell_node_ids(Γ_left)...))
fixed_dofs = vcat(2*left_nodes .- 1, 2*left_nodes)
free_dofs = setdiff(1:ndofs, fixed_dofs)

K_ff = K_global[free_dofs, free_dofs]
F_f  = F_global[free_dofs]

# -----------------------------
# Solve
# -----------------------------
U_low_API = zeros(ndofs)
U_low_API[free_dofs] = K_ff \ F_f;


In [79]:
# -------------------------------------------------
# 1. Prepare the right-edge boundary triangulation
# -------------------------------------------------
Γ = BoundaryTriangulation(model, tags="right")
face_node_ids = get_cell_node_ids(Γ)      # Vector{Vector{Int}} : nodes of each face
face_coords   = get_cell_coordinates(Γ)   # Vector{Vector{Point{2,Float64}}}

# -------------------------------------------------
# 2. 1-D reference element (linear edge)
# -------------------------------------------------
ref_nodes_line = Point{1,Float64}[Point(0.0), Point(1.0)]
m_line = MonomialBasis{1}(Float64,1)                # {1,ξ}
l_line = LagrangianDofBasis(Float64, ref_nodes_line) # N1=1-ξ, N2=ξ
change_line = inv(evaluate(l_line, m_line))         # change-of-basis matrix
s_line = linear_combination(change_line, m_line)    # shape functions in monomial basis

q_line = [Point{1,Float64}((0.5,))]   # Gauss point (ξ = 0.5)
w_line = [1.0]                       # weight (∫₀¹ dξ = 1)

t = SVector(0.0, -1.0)               # traction vector (constant)

# -------------------------------------------------
# 3. Element-by-element assembly
# -------------------------------------------------
# Container for *all* local element vectors (one per face)
local_Fe = [zeros(4) for _ in 1:length(face_node_ids)]

for (iface, nodes) in enumerate(face_node_ids)
    X_face = face_coords[iface]                 # [X₁, X₂]
    J      = norm(X_face[2] - X_face[1])        # edge length

    # Shape-function values at the Gauss point
    N_vals = [evaluate(s_line[i], q_line[1]) for i in 1:2]  # [N₁(ξ), N₂(ξ)]

    Fe = local_Fe[iface]                        # reference to the local vector
    fill!(Fe, 0.0)

    for i in 1:2                                # loop over the two edge nodes
        N = N_vals[i]
        Fe[2i-1] = t[1] * N * J * w_line[1]     # x-contribution
        Fe[2i  ] = t[2] * N * J * w_line[1]     # y-contribution
    end
end

# -------------------------------------------------
# 4. Scatter local contributions → global vector
# -------------------------------------------------
F_global = zeros(ndofs)

function add_to_global!(Fglob, Fe, nodes)
    for (i, node) in enumerate(nodes)
        dof_x = 2*node - 1
        dof_y = 2*node
        Fglob[dof_x] += Fe[2i-1]
        Fglob[dof_y] += Fe[2i  ]
    end
end

for (iface, nodes) in enumerate(face_node_ids)
    add_to_global!(F_global, local_Fe[iface], nodes)
end

# -------------------------------------------------
# 5. Dirichlet BC (left edge fixed) – unchanged
# -------------------------------------------------
Γ_left   = BoundaryTriangulation(model, tags="left")
left_nodes = unique(vcat(get_cell_node_ids(Γ_left)...))
fixed_dofs = vcat(2*left_nodes .- 1, 2*left_nodes)
free_dofs  = setdiff(1:ndofs, fixed_dofs)

K_ff = K_global[free_dofs, free_dofs]
F_f  = F_global[free_dofs]

# -------------------------------------------------
# 6. Solve
# -------------------------------------------------
U_low_API = zeros(ndofs)
U_low_API[free_dofs] = K_ff \ F_f

21960-element Vector{Float64}:
 -8.596841091664918e-5
 -5.7403255660628626e-5
 -0.00013342355691374052
 -8.335084240946604e-5
 -0.00019259816548575006
 -0.00011622528435026823
 -0.00023808489081530832
 -0.00013876533694841815
 -0.00029146910528249625
 -0.00016687309583429075
  ⋮
 -0.018990694868151713
  0.004590561582714269
 -0.019155869833401167
  0.00459483308509759
 -0.019322164611345213
  0.00459929034268874
 -0.019490997454258828
  0.004603607212224329
 -0.01966847963228987

In [80]:
# -----------------------------
L, W = 60.0, 20.0
partition = (Int(L*3), Int(W*3))
model = CartesianDiscreteModel((0.0,L,0.0,W), partition)

labels = get_face_labeling(model)
add_tag_from_tags!(labels, "left", [1,3,7])
add_tag_from_tags!(labels, "right", [2,4,8])

degree = 1
Ω = Triangulation(model)
dΩ = Measure(Ω, degree)
Γ = BoundaryTriangulation(model, tags="right")
dΓ = Measure(Γ, degree)

# -----------------------------
# FE space
# -----------------------------
order = 1
reffe = ReferenceFE(lagrangian, VectorValue{2,Float64}, order)
Vh = TestFESpace(Ω, reffe; conformity=:H1, dirichlet_tags="left")
Uh = TrialFESpace(Vh)
# -----------------------------
# Plane stress constitutive matrix (component-wise)
# -----------------------------
# 2D plane stress Lamé parameters
λ = E*ν/(1-ν^2)
μ = E/(2*(1+ν))
ε₀(u) = 0.5 * ( ∇(u) + transpose(∇(u))) # Strain; In Gridap this can be automatically defined in Gridap.ε
σ(ε₀) = λ * tr(ε₀) * one(ε₀) + 2 * μ * ε₀
# The weak form
a(u,v) = ∫((σ∘ε₀(u)) ⊙ ε₀(v))*dΩ # Left-hand size; (∘) Composite functions
Fvec(x) = VectorValue(0.0, -1.0)
l2(v) = ∫(Fvec ⋅ v) * dΓ
# -----------------------------
# Solve
# -----------------------------
op = AffineFEOperator(a, l2, Uh, Vh)
uh_high = solve(op)

SingleFieldFEFunction():
 num_cells: 10800
 DomainStyle: ReferenceDomain()
 Triangulation: BodyFittedTriangulation()
 Triangulation id: 9458263746817864554

In [81]:
U_high_API = get_free_dof_values(uh_high) # displacement vector
println("Norm of uh in high API = ", norm(U_high_API), "\n")
println("Norm of uh in low API = ", norm(U_low_API))


Norm of uh in high API = 1.035825683671591

Norm of uh in low API = 1.0358256836466653
